# A policy changes. A trend continues.

**A guided investigation · Chapter 5 · 30–45 minutes · No Python experience needed**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmg777/intro2causal/blob/main/notebooks_colab/explore-deaths_clean.ipynb)

> **The question**  
> How do drinking-age policy and young-adult mortality move across states and years?

When states change a drinking-age law, many other things are changing too: road safety, travel, economic conditions, and public attitudes. A state-by-year panel allows us to follow these overlapping changes rather than compare just two national averages. This prepared teaching extract contains death rates for young adults, a legal-access measure, and population information, repeated for several causes of death. Each row describes one state, year, and cause, not one person. The chapter identifies the population as ages 18–20, although the repository does not reproduce the entire upstream cleaning process. We will keep causes separate, follow selected state trajectories, and put policy changes alongside mortality trends. The point is to see why an apparent policy association raises further questions about timing and comparison groups before it becomes a causal claim.

### By the end, you will be able to…

- Read a panel as repeated state observations and investigate policy timing without pooling incompatible outcomes.
- Read and modify three interactive charts using short Python examples.
- Explain one reason a descriptive pattern alone cannot settle this case’s causal question.

### Before you see the evidence

If mortality falls after a law changes, what other trend could explain the decline?

Write two sentences now. You will return to them at the end.

**Your route:** meet the data → inspect three views → modify and try → ask the causal question.

## Your first few minutes in Colab · 3 minutes

You do not need prior Python experience. A **cell** is one block of text or code. Read the text, then press the **▶** beside the next code cell (or Shift+Enter). Its output appears underneath. Run cells in order: later cells use tables created earlier.

Choose **File → Save a copy in Drive** if you want to keep your changes. If a name is “not defined,” run the earlier setup cells again. **Runtime → Restart session and run all** gives you a fresh start. An error does not mean you have damaged the data: this notebook only reads it.

Keep a short learning log in a text cell. Before each chart, make a prediction. Afterward, write one observation with a number and one question that the chart cannot answer.

**Chart controls:** hover for values, click legend labels to hide/show a group, drag to zoom, and double-click to reset. A chart’s menu also offers a reset/autoscale control.

The setup installs Plotly Express if needed. `import` makes a library available; `pd` and `px` are short names for pandas (tables) and Plotly Express (charts).

In [ ]:
%pip install -q "plotly>=6,<7"

In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import display

# One consistent chart style; no plotting experience needed.
px.defaults.template = "plotly_white"
px.defaults.color_continuous_scale = "Blues"
px.defaults.color_discrete_sequence = ["#2563eb", "#7c3aed", "#059669", "#64748b"]

## Meet the data · 5 minutes

**One row:** State × year × death cause, for the chapter’s ages 18–20 population.  
**Stored size:** 4,284 rows × 7 columns.

| Column | What it records |
|---|---|
| `year` | Calendar year of the observation; census year in the child-labor file. |
| `state` | State identifier. Empirical state-panel codes follow U.S. state coding; simulated state IDs have no real geography. |
| `dtype` | Cause of death: all (all causes), MVA (motor vehicles), suicide, or internal. Filter before averaging rates. |
| `mrate` | Mortality rate per 100,000 in the chapter’s ages 18–20 population. |
| `legal` | Share of the relevant age group legally allowed to drink, from 0 to 1. |

Select one dtype before averaging rates. legal measures the share legally allowed to drink, not a randomized binary assignment. Population weighting changes which states contribute most.

`read_csv` downloads a table. We call it `df`. `head(5)` shows just five rows, not a separate sample used for the analysis. Square brackets select columns by their names.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/cmg777/intro2causal/main/data/ch5/deaths_clean.csv"
df = pd.read_csv(DATA_URL)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")
display(df[['year', 'state', 'dtype', 'mrate', 'legal']].head(5))

### Pause: what do the rows mean?

Choose one displayed row. Translate it into a sentence using the variable guide. Which values are categories or identifiers? Which are quantities? A numeric storage type does not necessarily make a code a meaningful measurement.

The next table checks types, missing values, and the number of distinct values for the key columns. `isna()` marks missing cells; `sum()` counts them. Missing is not zero. Ask which comparison would lose a row if one of its required values were missing.

In [ ]:
key_columns = ['year', 'state', 'dtype', 'mrate', 'legal']
checks = pd.DataFrame({"storage_type": df[key_columns].dtypes.astype(str),
                       "missing_rows": df[key_columns].isna().sum(),
                       "distinct_values": df[key_columns].nunique()})
display(checks)
print("The charts below omit missing values required for each plotted quantity.")

### A small summary before plotting

`describe()` summarizes numeric columns: `count` counts available values, `mean` is the average, and `50%` is the median. The median splits ordered values into two halves. Averages of identifiers or arbitrary category codes are not useful descriptions, even if Python calculates them.

For category-like key variables, the second table counts values, including missing entries. Check these labels before interpreting a chart.

In [ ]:
numeric_keys = df[key_columns].select_dtypes(include="number").columns
if len(numeric_keys):
    display(df[numeric_keys].describe().round(2))
for column in key_columns:
    if df[column].nunique() <= 8:
        display(df[column].value_counts(dropna=False).rename("Rows").to_frame())

## Read the chart code · 2 minutes

`px.bar`, `px.scatter`, `px.line`, and `px.box` choose a chart type. `x` and `y` name columns for the axes; `color` distinguishes groups or values. `labels` translates column names for readers. `fig.show()` displays the result.

A table transformation and a plot answer different questions: first decide what each plotted row represents, then choose how to display it. Before running each chart, predict what you expect to see.

In [ ]:
# Display labels: run once; you do not need to memorize this mapping.
labels = {'ability_group': 'Ability group',
 'census': 'Census year',
 'group': 'State group',
 'legal': 'Share legally allowed to drink',
 'mrate': 'Deaths per 100,000 (ages 18–20)',
 'offer': 'Scholarship assignment',
 'plan': 'Assigned insurance plan',
 'state': 'State code',
 'state_name': 'Simulated state',
 'year': 'Year'}

## Visual investigation 1 · Follow the same states over time

**Predict:** Do state mortality trends look alike?

We select all-cause deaths before comparing rates. The three state codes below are illustrative comparisons, not a selected treatment and control design. Sorting by year keeps lines in time order.

In [ ]:
allcause = df[df["dtype"] == "all"].copy()
selected = allcause[allcause["state"].isin([1, 6, 48])].sort_values("year")
selected["state_label"] = selected["state"].map({1: "Alabama", 6: "California", 48: "Texas"})
fig = px.line(selected, x="year", y="mrate", color="state_label", markers=True, labels={"year": "Year", "mrate": "Deaths per 100,000 (ages 18–20)", "state_label": "State"}, title="Different places, different trajectories")
fig.show()

**Explore the chart:** Toggle a state in the legend. Look for both level differences and different slopes.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

A shared downward trend could make a before/after policy comparison misleading. The chosen states illustrate heterogeneity; they are not automatically valid controls.

</details>

## Visual investigation 2 · Put policy timing on the page

**Predict:** When does legal access change in these same states?

legal is a share between 0 and 1. Multiplying by 100 displays a percentage. It measures policy access for the age group, not whether each person drinks.

In [ ]:
selected["legal_percent"] = 100 * selected["legal"]
fig = px.line(selected, x="year", y="legal_percent", color="state_label", markers=True, labels={"year": "Year", "legal_percent": "Share legally allowed to drink (%)", "state_label": "State"}, title="Policy timing differs too")
fig.update_yaxes(range=[0, 100])
fig.show()

**Explore the chart:** Locate a policy change and compare its year with the mortality chart. Does visual coincidence settle the causal question?

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

Changes at different dates can help a panel design, but policy choices may respond to circumstances that also affect mortality.

</details>

## Visual investigation 3 · A pooled pattern can hide a panel

**Predict:** What disappears when time and place are pooled?

Each dot is now a state-year for all-cause mortality. Color shows calendar year, so we can see whether a simple policy association also reflects time.

In [ ]:
fig = px.scatter(allcause, x="legal", y="mrate", color="year", hover_data=["state"], opacity=0.5, labels=labels, title="One scatter plot mixes many comparisons")
fig.show()

**Explore the chart:** Hover to identify the state and year behind a point. Check whether early and late years occupy different regions.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

A pooled association combines within-state changes, between-state differences, and time trends. The chapter’s panel design addresses a more specific comparison.

</details>

## Make it yours · 8 minutes

### 1. Modify an example

Rebuild selected from dtype == "MVA" and repeat the state trend chart; keep its cause label accurate.

Copy that chart cell before editing, so you can compare the original and modified outputs. Say what changed in your question as well as in your code.

### 2. Try a small investigation

Plot the unweighted yearly mean of all-cause state death rates and explain what receives equal weight.

Write a prediction first. The optional starter cell below is safe to leave unchanged; later cells do not depend on your answer.

In [ ]:
# Your investigation goes here.
# Describe your expected pattern before writing code.

<details>
<summary>Hint 1 · choose the right evidence</summary>

Start with allcause, not the table containing all causes together.

</details>

<details>
<summary>Hint 2 · a Python starting point</summary>

Use allcause.groupby("year", as_index=False)["mrate"].mean().

</details>

<details>
<summary>Optional worked solution · open after trying</summary>

<pre><code>yearly = allcause.groupby(&quot;year&quot;, as_index=False)[&quot;mrate&quot;].mean()
fig = px.line(yearly, x=&quot;year&quot;, y=&quot;mrate&quot;, markers=True, labels=labels, title=&quot;Yearly mean: every available state has equal weight&quot;)
fig.show()</code></pre>

The mean gives each state row equal weight, not each person. A population-weighted rate answers a different descriptive question.

Copy the code into the starter cell to run it.

</details>

## Optional extension · do weights change the picture?

First restrict to all-cause mortality. Population weighting gives larger age-group populations more influence; it answers a different question from an equal-state mean.

A **weighted mean** multiplies each value by its weight, adds those products, and divides by the sum of weights. The code below drops rows missing any required value and keeps positive weights. Each group gets its own denominator. This is a descriptive calculation, not an uncertainty estimate or a causal adjustment.

Predict whether weighting will change the comparison. Then compare the two summaries and explain which units receive more influence.

In [ ]:
weighted_data = df[df["dtype"] == "all"].dropna(subset=["mrate", "year", "pop"]).copy()
weighted_data = weighted_data[weighted_data["pop"] > 0]
weighted_data["product"] = weighted_data["mrate"] * weighted_data["pop"]
totals = weighted_data.groupby("year")[["product", "pop"]].sum()
weighted_mean = totals["product"] / totals["pop"]
unweighted_mean = weighted_data.groupby("year")["mrate"].mean()
comparison = pd.DataFrame({"Equal row weights": unweighted_mean, "Source weights": weighted_mean})
display(comparison.round(3))

## What can you now say? · 4 minutes

Return to your initial prediction. Write a four-sentence evidence note:

1. **I expected…** State your prediction.
2. **I observed…** Use a chart, a comparison, and the correct units.
3. **This does not establish…** Name a causal claim the evidence cannot settle.
4. **To learn more, I would need…** Propose an additional measurement or a more credible comparison.

### Carry this question into the chapter

**How could states with different policy timing help distinguish a policy effect from a shared national trend?**

[Continue with Chapter 5: Differences-in-Differences](https://cmg777.github.io/intro2causal/book/_book/notebooks_quarto/05-differences-in-differences.html#case-study-mlda-and-death-rates) · [Return to the case collection](https://cmg777.github.io/intro2causal/data-dictionary.html#ch5-deaths-clean-csv)

### Source and preparation notes

Chapter 5 MLDA teaching panel; Du Mouchel, Williams and Zador (1987), Raising the Alcohol Purchase Age: Its Effects on Fatal Motor Vehicle Crashes in Twenty-Six States (citation only; no verified destination included)

Prepared state panel derived from published MLDA research data. `pop` is an analysis weight, not used for the unweighted descriptive profiles here; the exact upstream transformation cannot be independently established from files in this repository.

These exploratory charts are based on the committed CSV. They are not a complete replication of the original study. Keep the source study, this prepared extract, and your descriptive results distinct.